# EduMentor AI — Module 1: Data Preprocessing
### GenAI & ML Bootcamp — Capstone Project

This notebook:
1. Loads the raw student dataset
2. Performs Exploratory Data Analysis (EDA)
3. Handles missing values
4. Encodes categorical variables
5. Normalizes numeric features
6. Saves the clean dataset for model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded ✓')

## 1. Load Raw Dataset

In [ ]:
df = pd.read_csv('../data/raw_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe(include='all')

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print('=== Target Variable Distribution ===')
print(df['final_result'].value_counts())
print(f'\nClass balance:\n{df["final_result"].value_counts(normalize=True).round(3)}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('EDA — Key Feature Distributions', fontsize=14, fontweight='bold')

# Target class distribution
df['final_result'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#4CAF50','#f44336','#FF9800','#2196F3'])
axes[0,0].set_title('Final Result Distribution')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=0)

# Age distribution
df['age'].hist(ax=axes[0,1], bins=15, color='#5C6BC0', edgecolor='white')
axes[0,1].set_title('Age Distribution')

# Average score distribution
df['avg_score'].hist(ax=axes[0,2], bins=20, color='#26A69A', edgecolor='white')
axes[0,2].set_title('Average Score Distribution')

# Days active distribution
df['days_active'].hist(ax=axes[1,0], bins=20, color='#FFA726', edgecolor='white')
axes[1,0].set_title('Days Active Distribution')

# Gender distribution
df['gender'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%', colors=['#42A5F5','#EF5350'])
axes[1,1].set_title('Gender Distribution')

# Education level
df['education_level'].value_counts().plot(kind='barh', ax=axes[1,2], color='#AB47BC')
axes[1,2].set_title('Education Level')

plt.tight_layout()
plt.savefig('../data/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA charts saved ✓')

In [ ]:
# Boxplot: Engagement by outcome
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Engagement Metrics by Final Result', fontsize=13, fontweight='bold')

for ax, col in zip(axes, ['avg_score', 'days_active', 'forum_posts']):
    df.boxplot(column=col, by='final_result', ax=ax)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Engagement by Outcome', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Handle Missing Values

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✓')

In [ ]:
df_clean = df.copy()

# Fill numeric missing with median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().any():
        median = df_clean[col].median()
        df_clean[col].fillna(median, inplace=True)
        print(f'  Filled {col} with median={median}')

# Fill categorical missing with mode
cat_cols = df_clean.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_clean[col].isnull().any():
        mode = df_clean[col].mode()[0]
        df_clean[col].fillna(mode, inplace=True)
        print(f'  Filled {col} with mode={mode}')

print(f'\nMissing after cleanup: {df_clean.isnull().sum().sum()} ✓')

## 4. Encode Categorical Variables

In [ ]:
# Create target: at_risk (1 = Fail or Withdrawn, 0 = Pass)
df_clean['at_risk'] = df_clean['final_result'].apply(
    lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
)
print(f'Target distribution:\n{df_clean["at_risk"].value_counts()}')

# Encode gender
df_clean['gender_encoded'] = df_clean['gender'].map({'M': 1, 'F': 0})

# Encode education level
edu_map = {
    'Lower Than A Level': 0,
    'A Level or Equivalent': 1,
    'HE Qualification': 2,
    'Post Graduate Qualification': 3
}
df_clean['edu_level_encoded'] = df_clean['education_level'].map(edu_map).fillna(1).astype(int)

# Encode disability
df_clean['disability_encoded'] = df_clean['disability'].map({'Y': 1, 'N': 0})

# Encode region
region_map = {
    'South Region': 0, 'North Region': 1,
    'East Region': 2, 'West Region': 3
}
df_clean['region_encoded'] = df_clean['region'].map(region_map).fillna(0).astype(int)

print('\nEncoding complete ✓')

## 5. Normalize Numeric Features

In [ ]:
# Normalize avg_score to [0, 1] if it's in percentage form
if df_clean['avg_score'].max() > 1.0:
    df_clean['avg_score'] = df_clean['avg_score'] / 100.0
    print('avg_score normalized to [0, 1] ✓')

# MinMax scale other continuous features
scale_cols = ['days_active', 'forum_posts', 'resource_clicks', 
              'assignment_submissions', 'video_views', 'quiz_attempts', 'age']

scaler = MinMaxScaler()
df_clean[scale_cols] = scaler.fit_transform(df_clean[scale_cols])

print(f'Scaled features: {scale_cols}')
df_clean[['avg_score'] + scale_cols].describe().round(3)

## 6. Final Feature Set & Save

In [ ]:
# Select final columns
feature_cols = [
    'student_id',
    'age', 'gender_encoded', 'edu_level_encoded', 'num_prev_attempts',
    'studied_credits', 'disability_encoded', 'region_encoded',
    'avg_score', 'days_active', 'forum_posts', 'resource_clicks',
    'assignment_submissions', 'video_views', 'quiz_attempts',
    'at_risk'
]

df_final = df_clean[feature_cols].copy()

print(f'Final dataset shape: {df_final.shape}')
print(f'\nClass distribution (at_risk):')
print(df_final['at_risk'].value_counts())
print(f'\nClass balance: {df_final["at_risk"].value_counts(normalize=True).round(3).to_dict()}')

df_final.head()

In [ ]:
# Correlation heatmap
numeric_features = [c for c in feature_cols if c != 'student_id']

plt.figure(figsize=(12, 8))
corr_matrix = df_final[numeric_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation heatmap saved ✓')

In [ ]:
# Save clean data
df_final.to_csv('../data/clean_data.csv', index=False)
print(f'✅ clean_data.csv saved with {df_final.shape[0]} rows and {df_final.shape[1]} columns')
print('\nPreprocessing complete! Ready for model training.')